# 🔬 Feature Engineering Example

Fit feature transformers on training data and apply to test data.

## Scenario
1. **Fit**: Learn mean/std from `train_data.csv`, categories for OneHot
2. **Transform**: Apply learned parameters to `test_data.csv` via `state_path`

> **📌 Best Practice Note**: This example uses `state_path` to ensure the test data is transformed using the same parameters learned from the training data (Fit on Train, Transform on Test). This prevents data leakage and ensures consistency between training and inference.

In [ ]:
!pip install -q "mlprep-rust==0.3.1" pandas pyarrow

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path.cwd() / 'outputs'
BASE.mkdir(exist_ok=True)

np.random.seed(42)
cities = ['Tokyo', 'Osaka', 'Nagoya', 'Fukuoka']

train = pd.DataFrame({
    'id': range(1, 101),
    'age': np.random.randint(20, 60, size=100),
    'income': np.random.randint(300, 1000, size=100) * 10000,
    'city': np.random.choice(cities, size=100)
})
test = pd.DataFrame({
    'id': range(101, 121),
    'age': np.random.randint(20, 60, size=20),
    'income': np.random.randint(300, 1000, size=20) * 10000,
    'city': np.random.choice(cities, size=20)
})

train.to_csv('train_data.csv', index=False)
test.to_csv('test_data.csv', index=False)
print('Generated train_data.csv and test_data.csv')
print(f'Train: {train.shape}, Test: {test.shape}')

In [ ]:
# Train pipeline: Fits transformers and SAVES state to feature_state.json
pipeline_train = '''inputs:
  - path: train_data.csv
    format: csv

steps:
  - type: features
    config:
      features:
        - column: age
          transform: standard_scale
        - column: income
          transform: standard_scale
        - column: city
          transform: one_hot_encode
    state_path: feature_state.json

outputs:
  - path: outputs/train_features.parquet
    format: parquet
'''

# Test pipeline: LOADS state from feature_state.json (uses Train's statistics)
pipeline_test = '''inputs:
  - path: test_data.csv
    format: csv

steps:
  - type: features
    config:
      features:
        - column: age
          transform: standard_scale
        - column: income
          transform: standard_scale
        - column: city
          transform: one_hot_encode
    state_path: feature_state.json

outputs:
  - path: outputs/test_features.parquet
    format: parquet
'''

with open('pipeline_train.yaml', 'w') as f:
    f.write(pipeline_train)
with open('pipeline_test.yaml', 'w') as f:
    f.write(pipeline_test)
print('Created pipeline_train.yaml and pipeline_test.yaml')
print('\n📌 Both pipelines share state_path: feature_state.json')
print('   - Train: Fits and saves state')
print('   - Test: Loads and applies saved state')

## 🚀 Run Train Pipeline

Fit transformers on training data and save state.

In [ ]:
!mlprep run pipeline_train.yaml --streaming --memory-limit 1GB

## 🚀 Run Test Pipeline

Apply transformations using the saved state from training.

> This ensures test data is scaled using Train's mean/std, not its own.

In [ ]:
!mlprep run pipeline_test.yaml --streaming --memory-limit 1GB

In [ ]:
import os

if os.path.exists('outputs/train_features.parquet'):
    train_feat = pd.read_parquet('outputs/train_features.parquet')
    print(f'✅ Train Features: {train_feat.shape}')
    print(train_feat.head())
else:
    print('❌ train_features.parquet not found')

if os.path.exists('outputs/test_features.parquet'):
    test_feat = pd.read_parquet('outputs/test_features.parquet')
    print(f'\n✅ Test Features: {test_feat.shape}')
    print(test_feat.head())
else:
    print('\n❌ test_features.parquet not found')

In [ ]:
if os.path.exists('feature_state.json'):
    print('✅ Feature state saved to feature_state.json')
    with open('feature_state.json') as f:
        print(f.read()[:500] + '...')
else:
    print('⚠️ feature_state.json not found')